In [ ]:
################################################## TabulaStromal Figures generation script ###############################
######################################################### 下面开始用stream来做 tragectory分析         ######### 
import h5py
import numpy as np
import scanpy as sc
import scipy.sparse as sp_sparse
import os
import pandas as pd
import gzip
import time
import re
#from pyensembl import EnsemblRelease
import pwd
import itertools
#### scanpy parameters
import omicverse as ov
import sys
import pandas as pd
#sc.settings.verbosity = 3  # verbosity: errors (0), warnings (1), info (2), hints (3)
#sc.logging.print_header()
#sc.settings.set_figure_params(dpi=80, facecolor="white")
import scvelo as scv
import cellrank as cr
from anndata import AnnData
import matplotlib.pyplot as plt
import dynamo as dyn
from scipy.stats import entropy
from sklearn.neighbors import NearestNeighbors
import stream as st


In [ ]:
Endo_meta = pd.read_csv('external_inputs/Endo_human_EndoDEG_count_notAge_633769.csv',sep=',')
LLM_meta = pd.read_csv('/fse/home/wangyongyan/Project/parser_code/result/2024_01_01_2025_08_17/gse_gsm_meta_rectify.csv',sep=',')
LLM_meta_hu = LLM_meta[LLM_meta['Species']=="['Homo sapiens']"]
Fibro_meta = pd.read_csv('./data/scalex/fibro/human/Fibro_human_FibDEG_count_notAge_GSMbatch_annotation_meta.tsv',sep='\t')



In [ ]:
LLM_meta_hu.columns

In [ ]:
LLM_meta_hu['tissue_final'].value_counts()

In [ ]:
# Endo_meta['tissue'].value_counts()


In [ ]:
# 排序（从大到小）
import math
lengths = Fibro_meta['tissue'].value_counts()
lengths.index = [lengths.index[i]+f" ({lengths.values[i]})" for i in range(len(lengths))]
lengths = lengths.map(math.log10).sort_values(ascending=True)

# 画水平条形图
fig, ax = plt.subplots(figsize=(4, 6))
lengths.plot.barh(ax=ax, color="#7895C1")

# 设置标签和标题
ax.set_xlabel("Count / log10")
ax.set_title("Fibro cellNum distribution", fontsize=14, fontweight="bold")
ax.set_ylabel("")

# 加粗边框
for spine in ax.spines.values():
    spine.set_linewidth(2)

plt.tight_layout()
plt.show()
plt.savefig('./TabulaStromal_figures/Fibro_tissue_cellNum_summary.png',  dpi=300, bbox_inches='tight', format='png')


In [ ]:
# 排序（从大到小）
import math
Fibro_meta_sample = Fibro_meta[['sample','tissue','age']].drop_duplicates() 
lengths = Fibro_meta_sample['tissue'].value_counts()
lengths.index = [lengths.index[i]+f" ({lengths.values[i]})" for i in range(len(lengths))]
lengths = lengths.map(math.log10).sort_values(ascending=True)

# 画水平条形图
fig, ax = plt.subplots(figsize=(4, 6))
lengths.plot.barh(ax=ax, color="#7895C1")

# 设置标签和标题
ax.set_xlabel("Count / log10")
ax.set_title("Fibro sample distribution", fontsize=14, fontweight="bold")
ax.set_ylabel("")

# 加粗边框
for spine in ax.spines.values():
    spine.set_linewidth(2)

plt.tight_layout()
plt.show()
plt.savefig('./TabulaStromal_figures/Fibro_tissue_sampleNum_summary.png',  dpi=300, bbox_inches='tight', format='png')
# plt.savefig(os.path.join( "./TabulaStromal_figures/Endo_tissue_sampleNum_summary.pdf"), dpi=300)


In [ ]:
############## To further check the annotation of adata object  ###### 
adata = sc.read_h5ad('./data/scalex/fibro/human/Fibro_human_FibDEG_count_notAge_GSMbatch_annotation.h5ad')
adata.shape


In [ ]:
###### Next need to further check the qFibro cells in some cluster 
qFibro_marker = ['APOD', 'FGF7', 'COL15A1','MFAP5', 'PI16', 'CD34']


In [ ]:
import scanpy as sc


sc.pl.dotplot(adata, var_names=qFibro_marker, groupby='leiden',swap_axes=True, show=True)
sc.pl.violin(adata, keys=qFibro_marker, groupby='leiden', show=True)

sc.pl.matrixplot(adata, var_names=qFibro_marker, groupby='leiden',swap_axes=True, show=True)

sc.pl.umap(adata, color=qFibro_marker, show=True,ncols=3)


In [ ]:
sc.tl.score_genes(adata, gene_list=qFibro_marker, score_name='qFibro')
sc.pl.umap(
    adata,
    color=['ECM','Immune','AP', 'qFibro'], 
           color_map='viridis', 
    size=2,  # 控制点的大小，默认可能是120，调小这个值使点更小
    alpha=0.5,  # 可选的透明度设置，使点更透明
    show=False,  # 不立即显示，方便后续调整
    frameon=False,  # 可选，去除边框
    legend_fontsize=12,  # 调整图例字体大小
    legend_fontoutline=2,  # 可选，图例字体轮廓
    ncols=2,
    vmin= 0, vmax= 3
)

# 如果你想进一步控制图形，可以使用matplotlib的API
import matplotlib.pyplot as plt
plt.gcf().set_size_inches(15, 10)  # 调整图形大小
plt.tight_layout()  # 调整布局
plt.show()


In [ ]:
############## 接下来画所有 Fibroblast marker 的 heatmap
fib_celltypes = adata.obs['Celltype'].unique()
fib_genelist = [celltype.split('_')[1] for celltype in fib_celltypes if len(celltype.split('_')) >= 2]

len(fib_genelist)


In [ ]:


existing_genes = [gene for gene in fib_genelist if gene in adata.var_names] + qFibro_marker

if existing_genes:
    # 3. 绘制高质量点图
    fig, ax = plt.subplots(figsize=(8, 14))
    sc.pl.dotplot(
        adata,
        var_names=existing_genes,
        groupby='Celltype_general',
        swap_axes=True,           # celltype在x轴，基因在y轴
        dot_max=0.8,              # 最大点大小
        cmap='viridis',           # 使用viridis颜色映射
        standard_scale='var',     # 按基因标准化表达值
        mean_only_expressed=True, # 只计算表达基因的平均值
        ax=ax,
        show=False
    )
    # 美化图形
    ax.set_title('Celltype Marker Gene Expression Profile', 
                fontsize=16, pad=25, fontweight='bold')
    ax.tick_params(axis='x', rotation=45)  # 旋转x轴标签
    plt.tight_layout()
    # 4. 保存多种格式
    # os.makedirs('results', exist_ok=True)
    
    # # 高分辨率PNG
    # plt.savefig('results/celltype_marker_expression.png', 
    #             dpi=300, bbox_inches='tight', facecolor='white')
    # # PDF格式
    # plt.savefig('results/celltype_marker_expression.pdf', 
    #             bbox_inches='tight', facecolor='white')
    plt.show()
    

In [ ]:
import pandas as pd
import numpy as np

# 方法3：严格处理，确保所有细胞都有器官信息
def strict_extract_organ(celltype):
    parts = celltype.split('_')
    if len(parts) < 3:
        # 如果格式不符合预期，记录并抛出错误
        raise ValueError(f"Celltype格式错误: '{celltype}'，无法提取器官信息")
    return parts[2]

try:
    adata.obs['Organ'] = adata.obs['Celltype'].apply(strict_extract_organ)
    print("器官信息成功添加到 adata.obs['Organ']")
    
    # 统计信息
    organ_counts = adata.obs['Organ'].value_counts()
    print("\n器官分布:")
    for organ, count in organ_counts.items():
        percentage = (count / len(adata.obs)) * 100
        print(f"  {organ}: {count} cells ({percentage:.1f}%)")
        
except ValueError as e:
    print(f"错误: {e}")
    print("请检查Celltype的命名格式")
    

In [ ]:
organ_list = [celltype.split('_')[2] for celltype in adata.obs['Celltype'] if len(celltype.split('_')) >= 2]
adata.obs['Organ'] = organ_list


In [ ]:
adata.obs['Organ'].value_counts()


In [ ]:
sorted(list(adata.obs['Celltype_sub'].unique()))



In [ ]:
# 确保基因列表中存在于数据集中的基因

# 计算signature score


In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns

# 定义Fibroblast分子特征基因集
# 根据图像中的信息定义4个主要的signature
signature_genes = {
    'ECM_Remodeling': [
        'COL1A1', 'COL1A2', 'COL3A1', 'COL6A1', 'COL6A2', 'COL6A3','FN1', 'LAMB1', 'LAMC1', 'SPARC', 'SPARC', 'TNC', 'VCAN','MMP2', 'MMP14', 'TIMP1', 'TIMP2', 'TIMP3', 'LOX', 'LOXL1', 'LOXL2'],
    'Myofibroblast': [
        'ACTA2', 'MYH11', 'TAGLN', 'CNN1', 'MYLK', 'MYL9','RGS5', 'NOTCH3', 'PDGFRB', 'DES', 'VIM'],
    'Immune_Regulatory': [
        'CCL2', 'CCL5', 'CXCL1', 'CXCL2', 'CXCL8', 'CXCL12','IL6', 'IL11', 'CSF1', 'CSF2', 'CSF3','ICAM1', 'VCAM1', 'SELE'],
    'Antigen_Presenting': [
        'HLA-A', 'HLA-B', 'HLA-C', 'HLA-DRA', 'HLA-DRB1', 'HLA-DPA1','HLA-DPB1', 'HLA-DQA1', 'HLA-DQB1', 'CD74', 'B2M','CIITA', 'CD80', 'CD86', 'CD40'
    ]
}


# 计算每个signature score
print("计算Fibroblast分子特征评分...")

for signature_name, genes in signature_genes.items():
    # 筛选出在数据集中存在的基因
    existing_genes = [gene for gene in genes if gene in adata.var_names]
    
    if len(existing_genes) > 0:
        print(f"计算 {signature_name} signature score (使用 {len(existing_genes)}/{len(genes)} 个基因)")
        # 使用scanpy的score_genes方法
        sc.tl.score_genes(adata, gene_list=existing_genes, score_name=signature_name)
    else:
        print(f"警告: {signature_name} 没有找到任何基因在数据集中")

# 创建三维散点图
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

# 获取三个主要的signature score作为三维坐标
x = adata.obs['ECM_Remodeling'].values  # X轴: ECM Remodeling
y = adata.obs['Myofibroblast'].values    # Y轴: Myofibroblast
z = adata.obs['Immune_Regulatory'].values # Z轴: Immune Regulatory

# 根据Celltype_sub着色
celltypes = adata.obs['Celltype_sub'].unique()
colors = plt.cm.tab20(np.linspace(0, 1, len(celltypes)))
color_dict = dict(zip(celltypes, colors))

# 为每个celltype绘制散点
for celltype in celltypes:
    mask = adata.obs['Celltype_sub'] == celltype
    ax.scatter(x[mask], y[mask], z[mask], 
               c=[color_dict[celltype]], 
               label=celltype, 
               s=20, 
               alpha=0.7)

# 设置坐标轴标签
ax.set_xlabel('ECM Remodeling\nScore', fontsize=12, labelpad=10)
ax.set_ylabel('Myofibroblast\nScore', fontsize=12, labelpad=10)
ax.set_zlabel('Immune Regulatory\nScore', fontsize=12, labelpad=10)

# 设置标题
ax.set_title('Fibroblast Lineage Molecular Signature Scores\n3D Visualization', 
             fontsize=14, pad=20)

# 添加图例
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)

# 调整视角
ax.view_init(elev=20, azim=45)

plt.tight_layout()
plt.show()

# # 保存图片
# plt.savefig('fibroblast_signature_3d.png', dpi=300, bbox_inches='tight')
# plt.savefig('fibroblast_signature_3d.pdf', bbox_inches='tight')



In [ ]:
adata_df = adata.obs[['leiden','MS_Fib_deg','T', 'B', 'Fibro', 'Endo',
       'MoMac', 'Epi', 'ECM', 'Immune', 'AP','qFibro']]
columns_to_avg = ['MS_Fib_deg','T', 'B', 'Fibro', 'Endo',
       'MoMac', 'Epi', 'ECM', 'Immune', 'AP','qFibro']  # 注意：根据实际列名调整
df1 = adata_df.groupby('leiden')[columns_to_avg].mean()
df1 = df1.reset_index()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

columns_to_plot = ['T', 'B', 'Fibro', 'Endo','MoMac', 'Epi', 'ECM', 'Immune', 'AP','qFibro']


# 计算全局纵轴范围（所有数据的min和max）
y_min = df1[columns_to_plot].min().min() - 0.1  # 最小值减缓冲
y_max = df1[columns_to_plot].max().max() + 0.1  # 最大值加缓冲

# 设置图形大小和子图布局（2行 x 3列）
fig, axes = plt.subplots(2, 3, figsize=(25, 12))
fig.suptitle('Grouped Bar Charts by Leiden (Split into 6 Subplots)', fontsize=16)

# 每个子图绘制7个Leiden
bar_width = 0.1
leiden_per_subplot = 7

for i in range(6):
    ax = axes[i // 3, i % 3]  # 定位子图位置
    start_idx = i * leiden_per_subplot
    end_idx = start_idx + leiden_per_subplot
    sub_df = df1.iloc[start_idx:end_idx]  # 当前子图的数据
    
    x = np.arange(len(sub_df['leiden']))  # 横轴位置
    
    # 为每一列绘制柱状图
    for j, column in enumerate(columns_to_plot):
        ax.bar(x + j * bar_width, sub_df[column], width=bar_width, label=column)
    
    # 设置子图标题和横轴标签
    ax.set_title(f'Leiden {start_idx}-{min(end_idx-1,36)}')
    ax.set_xticks(x + (len(columns_to_plot) - 1) * bar_width / 2)
    ax.set_xticklabels(sub_df['leiden'])
    ax.set_ylim(y_min, y_max)  # 统一纵轴范围
    
    # 只在第一个子图添加图例
    if i == 2:
        ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1))

# 调整布局防止重叠
plt.tight_layout()
plt.subplots_adjust(top=0.9)  # 为总标题留出空间
plt.show()


In [ ]:
########## 获取 Zemin paper's Fibroblasts marker gene list 
leiden_genelist = {
    'c01': ['MYH11', 'ACTA2', 'TAGLN'],
    'c02': ['ADAMDEC1', 'CCL8', 'APOE'],
    'c03': ['APOD', 'FGF7', 'COL15A1'],
    'c04': ['MMP11', 'COL10A1', 'POSTN', 'LRRC15'],
    'c05': ['MFAP5', 'P116', 'CD34'],
    'c06': ['LIMCH1', 'A2M', 'ADH1B'],
    'c07': ['CXCL1', 'CXCL2', 'IL6'],
    'c08': ['RGS5', 'CD36', 'NOTCH3'],
    'c09': ['WSB1', 'DDX17', 'CTNNB1'],
    'c10': ['PRG4', 'CLU', 'CRTAC1'],
    'c11': ['HOPX', 'IGFBP5', 'TIMP1'],
    'c12': ['KRT18', 'SLP1', 'UPK3B', 'MSLN'],
    'c13': ['CEBPD', 'CLU', 'CTGF', 'HGF'],
    'c14': ['HSPA6', 'DNAJB1', 'MYC', 'AFT4'],
    'c15': ['CXCL14', 'VSTM2A', 'SOX6'],
    'c16': ['SFRP4', 'SFRP2', 'COMP'],
    'c17': ['RBP1', 'STAR', 'STMN1'],
    'c18': ['ACTG2', 'HHIP', 'CNN1'],
    'c19': ['MMP1', 'COLTA1', 'WNT5A', 'ISG15', 'IL7R'],
    'c20': ['CXCL12', 'CDT4', 'HLA-DRB1', 'HLA-DRA']
}

gene_list = list(set(gene for sublist in leiden_genelist.values() for gene in sublist))
# gene_list


In [ ]:
available_genes = [gene for gene in gene_list if gene in adata.var_names]
missing_genes = [gene for gene in gene_list if gene not in adata.var_names]
len(available_genes), len(missing_genes)

In [ ]:
os.getcwd()
# os.chdir('./TabulaStromal_figures')    ### 这样对存储有影响

In [ ]:
################# 接下来画 adata 中注释好后的gene list的表达 
Celltype_list = list(adata.obs['Celltype'].unique())
gene_list_heatmap = Celltype_list.split()


In [ ]:
adata.obs

In [ ]:
#### Split test & train set
tissue_dir = os.path.join(wang_dir, tissue)
target_dir = os.path.join('./data/scalex/fibro/human/age_prediction/intermediate/predictor_embedding/transfer', tissue)
train_dir = os.path.join('./data/scalex/fibro/human/age_prediction/intermediate/predictor_embedding/train', tissue)
test_dir = os.path.join('./data/scalex/fibro/human/age_prediction/intermediate/predictor_embedding/test', tissue)
os.makedirs(target_dir, exist_ok=True)
os.makedirs(train_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)
files = os.listdir(tissue_dir)
zfiles = random.sample(files, k=max(1, round(len(files)*test_percent)))
tfiles = [each for each in files if each not in zfiles]
for file in zfiles:
    shutil.copy(os.path.join(tissue_dir, file), os.path.join(target_dir, file))
for file in tfiles:
    print(file)
    data, age = torch.load(os.path.join(tissue_dir, file))
    train_data, test_data = train_test_split(data, test_size=test_percent, random_state=2025)
    train_data = [train_data, age]
    test_data = [test_data, age]
    torch.save(train_data, os.path.join(train_dir, file))
    torch.save(test_data, os.path.join(test_dir, file))
    

In [ ]:
########## Pseudo_individual.py  
if __name__ == "__main__":
    args = get_args()
    path_emb_data = args.path_emb_data
    ncells = args.ncells
    path_save = args.path_save
    tissues = os.listdir(path_emb_data)
    for tissue in tissues:
        tmp_path_save = os.path.join(path_save, tissue)
        os.makedirs(tmp_path_save, exist_ok=True)
        files = [os.path.join(path_emb_data, tissue, file) for file in os.listdir(os.path.join(path_emb_data, tissue))]
        output = []
        count = 0
        for e in range(10):
            random.shuffle(files)
            for file in tqdm.tqdm(files, ncols=80, desc=f'pseudo {tissue} round {e}'):
                data, age = torch.load(file)
                ninputs = round(data.shape[0] / 10)
                if ninputs > ncells*1.5:
                    psedual_individuals = [random.sample(list(range(len(data))), k=ncells) for _ in range(ninputs)]
                    psedual_individuals = [data[indices].mean(axis=0) for indices in psedual_individuals]
                    output += [[each, age] for each in psedual_individuals]
                else:
                    psedual_individuals = data.mean(axis=0)
                    output += [[psedual_individuals, age]]
                if len(output) > 10000:
                    torch.save(output, os.path.join(tmp_path_save, f"{tissue}_pseudo_individuals_{count}.pt"))
                    count += 1
                    output = []
                    # break
        if output:
            torch.save(output, os.path.join(tmp_path_save, f"{tissue}_pseudo_individuals_{count}.pt"))
        # break
    print("Mission success!")

In [ ]:
########## Get the peseudotime data 
count = 0
for e in range(10):
    random.shuffle(files)
    for file in tqdm.tqdm(files, ncols=80, desc=f'pseudo {tissue} round {e}'):
        data, age = torch.load(file)
        ninputs = round(data.shape[0] / 10)
        if ninputs > ncells*1.5:
            psedual_individuals = [random.sample(list(range(len(data))), k=ncells) for _ in range(ninputs)]
            psedual_individuals = [data[indices].mean(axis=0) for indices in psedual_individuals]
            output += [[each, age] for each in psedual_individuals]
        else:
            psedual_individuals = data.mean(axis=0)
            output += [[psedual_individuals, age]]
        if len(output) > 10000:
            torch.save(output, os.path.join(tmp_path_save, f"{tissue}_pseudo_individuals_{count}.pt"))
            count += 1
            output = []
            # break
if output:
    torch.save(output, os.path.join(tmp_path_save, f"{tissue}_pseudo_individuals_{count}.pt"))


In [ ]:
############### Run the age predictor script 
individuals_path = os.path.join(data_path, tissue)
files = [os.path.join(individuals_path,each) for each in os.listdir(individuals_path)]
os.makedirs(os.path.join(save_path, tissue), exist_ok=True)
logging.basicConfig(
    filename=os.path.join(save_path, tissue, "log.txt"),
    filemode="w",
    level=logging.INFO,
    format="%(asctime)s - %(message)s",
)
logging.info(f"Parameters:\n{args}\n")

# --- Start Deep-lerning models training block ---
# device = get_free_gpu()
# device = torch.device(f"cuda:{device}")
# model = Predictor().to(device)
# criterion = nn.MSELoss()
# optimizer = optim.Adam(model.parameters(), lr=lr)
# scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
#     optimizer, 
#     mode='min',        # 目标是最小化 val_mae
#     factor=lr_decay,        # 学习率缩小一半
#     patience=lr_patience,        # 连续5个epoch无提升则调整LR
#     threshold=1e-4    # 改进小于这个数视为没有提升
# )
# early_stopping = EarlyStopping(patience=early_stop_patience, delta=1e-4, mode="min")  # 用验证集 MAE 监控

# best_val_mae = float('inf')
# for e in range(epoch):
#     val_total_data = []
#     test_total_data = []
#     train_total_loss = []
#     train_total_preds = []
#     train_total_targets = []

#     # train
#     for file in files:
#         data = torch.load(file)
#         train_data, temp_data = train_test_split(data, test_size=0.2, random_state=2025)
#         val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=2025)
#         val_total_data += val_data
#         test_total_data += test_data
#         train_dataset = IndividualDataset(train_data)
#         train_loader = torch.utils.data.DataLoader(
#             train_dataset, 
#             batch_size=batch_size, 
#             shuffle=True,
#             num_workers=2
#             )
#         train_loss, train_preds, train_targets = train(train_loader, model, criterion, optimizer, device)
#         train_total_loss += train_loss
#         train_total_preds += train_preds
#         train_total_targets += train_targets
#     train_metrics = regression_metrics(train_total_targets, train_total_preds)
#     train_avg_loss = round(np.mean(train_total_loss), 4)

#     # val
#     val_dataset = IndividualDataset(val_total_data)
#     val_loader = torch.utils.data.DataLoader(
#         val_dataset, 
#         batch_size=batch_size, 
#         shuffle=False,
#         num_workers=2
#         )
#     val_preds, val_targets = test(val_loader, model, device)
#     val_metrics = regression_metrics(val_preds, val_targets)
    
#     # log
#     current_lr = optimizer.param_groups[0]['lr']
#     logging.info(f"[Epoch {e+1}] "
#         f"Loss={train_avg_loss:.4f}, "
#         f"MAE={train_metrics['MAE']:.4f}, "
#         f"RMSE={train_metrics['RMSE']:.4f}, "
#         f"R2={train_metrics['R2']:.4f}, "
#         f"LR={current_lr:.4f}   "
#         f"   [Val] "
#         f"MAE={val_metrics['MAE']:.4f}, "
#         f"RMSE={val_metrics['RMSE']:.4f}, "
#         f"R2={val_metrics['R2']:.4f}")
    
#     # Dynamic LR 
#     val_mae = val_metrics['MAE']
#     scheduler.step(val_mae)  # ReduceLROnPlateau 监控 MAE
#     if val_mae < best_val_mae - 1e-4:
#         best_val_mae = val_mae
#         torch.save(model.state_dict(), os.path.join(save_path, tissue, "best_model.pth"))
#     # EarlyStopping
#     if early_stopping(val_mae):
#         logging.info(f"Early stopping triggered at epoch {e+1}")
#         break

# # test
# test_dataset = IndividualDataset(test_total_data)
# test_loader = torch.utils.data.DataLoader(
#     test_dataset, 
#     batch_size=batch_size, 
#     shuffle=False,
#     num_workers=2
#     )
# test_preds, test_targets = test(test_loader, model, device)
# test_preds = torch.cat(test_preds)
# test_targets = torch.cat(test_targets)
# torch.save(test_preds, os.path.join(save_path, tissue, "test_preds.pt"))
# torch.save(test_targets, os.path.join(save_path, tissue, "test_targets.pt"))
# test_metrics = regression_metrics(test_preds, test_targets)
# logging.info(f"   [Test] "
#     f"MAE={test_metrics['MAE']:.4f}, "
#     f"RMSE={test_metrics['RMSE']:.4f}, "
#     f"R2={test_metrics['R2']:.4f}")

# --- End deep-learning models training block --

# --- Start Decision-making tree models training block --
model = AgePredictorXGB()

train_total_data = []
val_total_data = []
test_total_data = []
for file in files:
    data = torch.load(file)
    train_data, temp_data = train_test_split(data, test_size=0.2, random_state=2025)
    val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=2025)
    train_total_data += train_data
    val_total_data += val_data
    test_total_data += test_data
train_data, train_label = list(zip(*train_total_data))
val_data, val_label = list(zip(*val_total_data))
test_data, test_label = list(zip(*test_total_data))
train_data = torch.stack(train_data)
train_label = torch.stack(train_label)
val_data = torch.stack(val_data)
val_label = torch.stack(val_label)
test_data = torch.stack(test_data)
test_label = torch.stack(test_label)

logging.info("Training start...")
model.fit(train_data, train_label)
model.save_model(os.path.join(save_path, tissue, "best_model.json"))
logging.info("Training complete!")
test_pred = model.predict(test_data)
test_metrics = regression_metrics(test_label, test_pred)
logging.info(f"   [Test] "
    f"MAE={test_metrics['MAE']:.4f}, "
    f"RMSE={test_metrics['RMSE']:.4f}, "
    f"R2={test_metrics['R2']:.4f}")

# --- End Decision-making tree models training block --

In [ ]:
##### According to the meta information to split adata objects into single individuals  #####
meta_data = pd.read_csv('./data/scalex/fibro/human/Fibro_human_FibDEG_count_notAge_GSMbatch_annotation_meta.tsv',sep='\t')



In [ ]:
################ Use the cellrank do the tragectory analysis 

#  adata = sc.read_h5ad('./data/scalex/fibro/human/stream_result/subtype_only/Fibro_general/bbknn/Fibro_human_FibDEG_count_notAge_GSMbatch_annotation_BatT_12608.h5ad')
adata = sc.read_h5ad('./data/scalex//endo/human/trajectory/data/EndoAtlas_CelltypePan_BatT_19011.h5ad')
# adata = sc.read_h5ad('./data/scalex/fibro/human/trajectory/data/FibAtlas_CelltypePan_BatT_47646.h5ad')


In [ ]:
adata.shape


In [ ]:
adata.obs['res2'] = adata.obs['res2'].astype(str)
adata.obs['Celltype_general'] = adata.obs['Celltype_general'].astype(str)
adata.obs['label'] = adata.obs['Celltype_general']
# adata.write('./data/scalex/fibro/human/trajectory/data/FibAtlas_CelltypePan_BatT_45050.h5ad')


In [ ]:
sc.settings.set_figure_params(dpi=80, frameon=False, figsize=(3, 3), facecolor="white")
adata.obs['leiden'] = adata.obs['res2']
ax = sc.pl.umap(adata, color=['leiden'],legend_loc='on data',show=False ) 
ax.set_title(
    'Leiden', 
    fontsize=20,              # 设置字体大小
    # fontweight='bold',          # 设置字体粗细
    # legend_fontsize=14,
    family='Times New Roman'    # 设置字体家族
)
plt.tight_layout()
fig_dir ='./TabulaStromal_figures/Fibroblasts/tragectory_fig/CellRank/'
# plt.savefig(
# fig_dir+"Fibro_CellRank_leiden.pdf",
# dpi=300, bbox_inches='tight',format='pdf')
# plt.savefig(fig_dir+"Fibro_CellRank_leiden.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# 定义quick_cellrank_analysis函数，估计初始细胞和终末状态
def quick_cellrank_analysis(adata):
    # 1. CellRank 核心
    import cellrank as cr
    ck = cr.kernels.ConnectivityKernel(adata)
    ck = ck.compute_transition_matrix()

    estimator = cr.estimators.GPCCA(ck)
    estimator.compute_schur(n_components=5)
    estimator.compute_macrostates(n_states=5)

    # 保存宏观状态
    adata.obs['macrostates'] = estimator.macrostates

    # 2. 识别初始细胞
    import pandas as pd
    initial_labels = pd.Series('', index=adata.obs.index)
    if 'louvain_integr' in adata.obs.columns:
        for cluster in adata.obs['louvain_integr'].unique()[:10]:  # 前10个聚类
            cluster_cells = np.where(adata.obs['louvain_integr'] == cluster)[0]
            if len(cluster_cells) > 0:
                initial_labels.iloc[cluster_cells[0]] = f'Initial_{cluster}'
    adata.obs['initial_cells'] = initial_labels

    # 3. 计算终末状态
    try:
        estimator.compute_absorption_probabilities()
        # 从吸收概率获取终末状态
        if hasattr(estimator, 'absorption_probabilities'):
            terminal_states = estimator.absorption_probabilities.idxmax(axis=1)
            adata.obs['terminal_states'] = terminal_states
    except:
        # 如果失败，使用宏观状态作为终末状态
        adata.obs['terminal_states'] = estimator.macrostates

    # 4. 可视化
    import matplotlib.pyplot as plt
    import scanpy as sc
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    # 初始细胞
    sc.pl.umap(adata, color='initial_cells', ax=axes[0],
               show=False, frameon=False, title='Initial Cells')
    # 宏观状态
    sc.pl.umap(adata, color='macrostates', ax=axes[1],
               show=False, frameon=False, title='Macro States')
    # 终末状态
    sc.pl.umap(adata, color='terminal_states', ax=axes[2],
               show=False, frameon=False, title='Terminal States')

    plt.tight_layout()
    plt.show()
    return estimator


In [ ]:
estimator = quick_cellrank_analysis(adata) 


In [ ]:
# sorted(dir())


In [ ]:
adata.shape


In [ ]:
# 计算谱系（lineages）和命运概率
def analyze_lineages(adata, estimator):
    lineages_data = None
    if 'macrostates' in adata.obs.columns:
        # 将宏观状态转换为 one-hot 编码
        macrostates = adata.obs['macrostates']
        # 创建谱系矩阵
        unique_states = [s for s in macrostates.unique() if pd.notna(s)]
        lineages_data = pd.DataFrame(0, index=adata.obs.index,
                                    columns=[f'Lineage_{s}' for s in unique_states])
        for i, state in enumerate(unique_states):
            mask = macrostates == state
            lineages_data.iloc[mask, i] = 1
    else:
        print("❌ 无法获取谱系信息")
        return None
    # 保存到 adata
    if lineages_data is not None:
        adata.obsm['X_lineages'] = lineages_data.values if hasattr(lineages_data, 'values') else lineages_data
        # 如果是 DataFrame，保存列名
        if isinstance(lineages_data, pd.DataFrame):
            adata.uns['lineages_names'] = lineages_data.columns.tolist()
        print(f"✅ 谱系数据已保存到 adata.obsm['X_lineages']")
        print(f"  谱系数量: {adata.obsm['X_lineages'].shape[1]}")
        return lineages_data
    return None


In [ ]:
# 使用
lineages = analyze_lineages(adata, estimator)


In [ ]:
# sorted(dir())
# adata.X.mean()



In [ ]:
def visualize_lineages(adata, lineages_data=None):
    import numpy as np
    # 获取谱系数据
    if lineages_data is None and 'X_lineages' in adata.obsm:
        lineages_data = adata.obsm['X_lineages']

    # 如果是数组，转换为 DataFrame
    if isinstance(lineages_data, np.ndarray):
        n_lineages = lineages_data.shape[1]
        lineages_data = pd.DataFrame(
            lineages_data,
            index=adata.obs.index,
            columns=[f'Lineage_{i+1}' for i in range(n_lineages)]
        )

    if lineages_data is None:
        print("⚠️  没有谱系数据可可视化")
        return

    # 创建可视化
    n_lineages = min(4, lineages_data.shape[1])  # 最多显示4个谱系
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    axes = axes.flatten()

    for i in range(min(n_lineages, 4)):
        ax = axes[i]

        # 获取当前谱系
        if isinstance(lineages_data, pd.DataFrame):
            lineage_name = lineages_data.columns[i]
            lineage_values = lineages_data.iloc[:, i]
        else:
            lineage_name = f'Lineage_{i+1}'
            lineage_values = lineages_data[:, i]

        # 在 UMAP 上可视化谱系概率
        scatter = ax.scatter(adata.obsm['X_umap'][:, 0],
                             adata.obsm['X_umap'][:, 1],
                             c=lineage_values,
                             cmap='viridis',
                             s=10,
                             alpha=0.8)

        ax.set_xlabel('UMAP1')
        ax.set_ylabel('UMAP2')
        ax.set_title(f'{lineage_name}')
        ax.grid(True, alpha=0.3)

        # 添加颜色条
        plt.colorbar(scatter, ax=ax, label='Probability')

    # 如果少于4个谱系，隐藏多余的子图
    for i in range(n_lineages, 4):
        axes[i].axis('off')

    plt.tight_layout()
    # plt.savefig('lineages_visualization.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("✅ 谱系可视化已保存为 'lineages_visualization.png'")


In [ ]:
# 使用
visualize_lineages(adata, lineages)


In [ ]:
# 计算伪时间
def quick_pseudotime(adata, estimator=None):
    """快速计算伪时间"""
    import scanpy as sc
    import numpy as np
    # 优先使用DPT
    try:
        sc.tl.diffmap(adata)

        # 选择根细胞
        if 'initial_cells' in adata.obs.columns:
            initial_mask = adata.obs['initial_cells'] != ''
            if initial_mask.any():
                root_cell = np.where(initial_mask)[0][0]
            else:
                root_cell = 0
        else:
            root_cell = 0

        adata.uns['iroot'] = root_cell
        sc.tl.dpt(adata)

        pseudotime = adata.obs['dpt_pseudotime'].values
        method = 'dpt'

    except:
        # 如果DPT失败，使用Schur向量
        if estimator is not None and hasattr(estimator, 'schur_vectors'):
            schur_vec = estimator.schur_vectors[:, 1]
            pseudotime = (schur_vec - schur_vec.min()) / (schur_vec.max() - schur_vec.min())
            method = 'schur_vec'
        else:
            # 最后使用UMAP距离
            umap_coords = adata.obsm['X_umap']
            distances = np.linalg.norm(umap_coords - umap_coords[0], axis=1)
            pseudotime = (distances - distances.min()) / (distances.max() - distances.min())
            method = 'umap_distance'

    adata.obs['pseudotime'] = pseudotime
    print(f"✅ 使用 {method} 计算伪时间，范围: [{pseudotime.min():.3f} - {pseudotime.max():.3f}]")

    return pseudotime


In [ ]:
# 使用
pseudotime = quick_pseudotime(adata, estimator)


In [ ]:
# 伪时间可视化
plt.figure(figsize=(8, 6))
plt.scatter(adata.obsm['X_umap'][:, 0], adata.obsm['X_umap'][:, 1],
            c=pseudotime, cmap='viridis', s=10, alpha=0.8)
plt.colorbar(label='Pseudotime')
plt.xlabel('UMAP1')
plt.ylabel('UMAP2')
plt.title('Pseudotime [Fibro]')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
def simple_trajectory_streamplot(adata):
    if 'dpt_pseudotime' in adata.obs.columns:
        pseudotime = adata.obs['dpt_pseudotime'].values
        coords = adata.obsm['X_umap']

        # 计算速度场（简化版）
        from scipy.interpolate import griddata
        import numpy as np

        xi = np.linspace(coords[:, 0].min(), coords[:, 0].max(), 100)
        yi = np.linspace(coords[:, 1].min(), coords[:, 1].max(), 100)
        xi_grid, yi_grid = np.meshgrid(xi, yi)

        # 计算伪时间梯度作为速度方向
        dx = griddata(coords, np.gradient(pseudotime, coords[:, 0]), (xi_grid, yi_grid), method='linear')
        dy = griddata(coords, np.gradient(pseudotime, coords[:, 1]), (xi_grid, yi_grid), method='linear')

        # 创建图
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))

        # 左图：伪时间
        scatter = axes[0].scatter(coords[:, 0], coords[:, 1],
                                  c=pseudotime, cmap='viridis',
                                  s=10, alpha=0.8)
        axes[0].set_xlabel('UMAP1')
        axes[0].set_ylabel('UMAP2')
        axes[0].set_title('Pseudotime')
        plt.colorbar(scatter, ax=axes[0], label='Pseudotime')

        # 右图：流线图
        axes[1].scatter(coords[:, 0], coords[:, 1],
                        c=pseudotime, cmap='viridis',
                        s=10, alpha=0.3)
        # 绘制流线
        axes[1].streamplot(xi_grid, yi_grid, dx, dy,
                          color='black', linewidth=1.5,
                          density=2, arrowstyle='->')

        axes[1].set_xlabel('UMAP1')
        axes[1].set_ylabel('UMAP2')
        axes[1].set_title('Trajectory Streamlines')

        plt.tight_layout()
        # plt.savefig('simple_streamplot.png', dpi=300, bbox_inches='tight')
        plt.show()
        print("✅ 简化版流线图已保存为 'simple_streamplot.png'")
    else:
        print("没有伪时间数据")

# 使用
simple_trajectory_streamplot(adata)


In [ ]:
sc.settings.set_figure_params(dpi=80, frameon=False, figsize=(3, 3), facecolor="white")
axes = sc.pl.umap(
    adata, 
    color=['Celltype_general', 'leiden', 'Celltype'],  
    show=False, 
    ncols=2, 
    wspace=0.3,
    hspace=0.2
)
# 调整所有子图的标题和图例文本
for ax in axes:
    # 调整标题
    ax.set_title(ax.get_title(), fontsize=20)   # , fontweight='bold'
    # 如果有图例，调整图例字体
    if ax.get_legend():
        plt.setp(ax.get_legend().get_texts(), fontsize=14)
    
    # 调整坐标轴标签字体
    ax.xaxis.label.set_size(14)
    ax.yaxis.label.set_size(14)
    # 调整刻度标签字体
    ax.tick_params(axis='both', labelsize=12)

plt.tight_layout()
fig_dir ='./TabulaStromal_figures/Fibroblasts/tragectory_fig/'
# plt.savefig(
# fig_dir+"Fibro_CellRank_umap.pdf",
# dpi=300, bbox_inches='tight',format='pdf')
# plt.savefig(fig_dir+"Fibro_CellRank_umap.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
adata.obs[['Celltype_general','leiden', 'Celltype']].drop_duplicates().sort_values('Celltype_general')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
from scipy.ndimage import gaussian_filter
from scipy.interpolate import griddata
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch

def smooth_trajectory_streamplot(
        adata,
        n_neighbors=30,
        grid_density=200,
        smooth_sigma=2,
        density=1.2,
        linewidth=2
):
    """
    更平滑、更长、更清晰的 trajectory streamplot
    """
    if 'dpt_pseudotime' not in adata.obs.columns:
        print("❌ 没有伪时间数据")
        return

    coords = adata.obsm['X_umap']
    pseudotime = adata.obs['dpt_pseudotime'].values
    # 1️⃣ 局部线性拟合计算方向
    nbrs = NearestNeighbors(n_neighbors=n_neighbors).fit(coords)
    _, indices = nbrs.kneighbors(coords)
    velocity = np.zeros_like(coords)
    for i, neighbors in enumerate(indices):
        X_local = coords[neighbors]
        y_local = pseudotime[neighbors]

        # 线性回归拟合：pseudotime = ax + by + c
        A = np.c_[X_local, np.ones(len(X_local))]
        coef, _, _, _ = np.linalg.lstsq(A, y_local, rcond=None)
        velocity[i] = coef[:2]  # (a, b)
    # 2️⃣ 创建高分辨率网格
    xi = np.linspace(coords[:, 0].min(), coords[:, 0].max(), grid_density)
    yi = np.linspace(coords[:, 1].min(), coords[:, 1].max(), grid_density)
    xi_grid, yi_grid = np.meshgrid(xi, yi)
    # 插值到规则网格
    dx = griddata(coords, velocity[:, 0], (xi_grid, yi_grid), method='cubic')
    dy = griddata(coords, velocity[:, 1], (xi_grid, yi_grid), method='cubic')
    # 3️⃣ 高斯平滑
    dx = gaussian_filter(dx, sigma=smooth_sigma)
    dy = gaussian_filter(dy, sigma=smooth_sigma)
    # 4️⃣ 画图
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0], axes[1] = axes
    # 左图：pseudotime
    scatter = axes[0].scatter(
        coords[:, 0], coords[:, 1],
        c=pseudotime,
        cmap='viridis',
        s=10,
        alpha=0.9
    )
    
    axes[0].set_title("Pseudotime", fontsize=16)
    axes[0].set_xlabel("UMAP1")
    axes[0].set_ylabel("UMAP2")

    axes[0].set_aspect("equal", adjustable="box")
    axes[1].set_aspect("equal", adjustable="box")
    
    divider = make_axes_locatable(axes[0])
    cax = divider.append_axes("right", size="3%", pad=0.05)
    cbar = plt.colorbar(scatter, cax=cax)
    cbar.set_label("Pseudotime", rotation=270, labelpad=15)
    cbar.ax.tick_params(labelsize=9)
    
    # plt.colorbar(scatter, ax=axes[0], label="Pseudotime")
    # 右图：流线
    axes[1].scatter(
        coords[:, 0], coords[:, 1],
        c=pseudotime,
        cmap='viridis',
        s=10,
        alpha=0.3
    )
    axes[1].streamplot(
        xi_grid,
        yi_grid,
        dx,
        dy,
        color='black',
        linewidth=linewidth,
        density=density,
        arrowsize=2,
        arrowstyle='-|>'
    )
    axes[1].set_title("Trajectory Streamlines", fontsize=16)
    axes[1].set_xlabel("UMAP1")
    axes[1].set_ylabel("UMAP2")
    
    plt.subplots_adjust(
        left=0.06,
        right=0.97,
        bottom=0.08,
        top=0.92,
        wspace=0.25
    )

    plt.tight_layout()
    fig_dir ='./TabulaStromal_figures/Fibroblasts/tragectory_fig/'
    # plt.savefig(
    # fig_dir+"Fibro_CellRank_tragectory.pdf",
    # dpi=300, bbox_inches='tight',format='pdf')
    # plt.savefig(fig_dir+"Fibro_CellRank_tragectory.png", dpi=300, bbox_inches="tight")
    plt.show()
    

In [ ]:
smooth_trajectory_streamplot(
    adata,
    n_neighbors=50,
    grid_density=250,
    smooth_sigma=3,
    density=0.9,
    linewidth=2.5
)


In [ ]:
cluster_directional_flow(
    adata,
    cluster_key="Celltype_general",
    pseudotime_key="dpt_pseudotime",
    min_cells=50,
    distance_threshold=10,
    arrow_scale=20
)



In [ ]:
help(cluster_directional_flow_optimized)

In [ ]:
def cluster_directional_flow_optimized(
        adata,
        cluster_key="Celltype_general",
        pseudotime_key="dpt_pseudotime",
        min_cells=30,
        distance_threshold=8,
        arrow_scale=15,
        figsize=(14, 6)
):
    """
    优化版 cluster-level directional flow
    - UMAP 占满画布
    - 左右比例完全一致
    - colorbar 不压缩主图
    """
    coords = adata.obsm["X_umap"]
    pseudotime = adata.obs[pseudotime_key].values
    clusters = adata.obs[cluster_key]
    df = pd.DataFrame({
        "x": coords[:, 0],
        "y": coords[:, 1],
        "pseudotime": pseudotime,
        "cluster": clusters.values
    })
    # ----------------------------
    # 1️⃣ 计算 cluster stats
    # ----------------------------
    cluster_stats = (
        df.groupby("cluster")
        .agg(
            x_mean=("x", "mean"),
            y_mean=("y", "mean"),
            pt_mean=("pseudotime", "mean"),
            size=("cluster", "size")
        )
        .reset_index()
    )
    cluster_stats = cluster_stats[cluster_stats["size"] > min_cells]
    # ----------------------------
    # 2️⃣ 创建共享坐标子图
    # ----------------------------
    fig, axes = plt.subplots(
        1, 2,
        figsize=figsize,
        sharex=True,
        sharey=True
    )

    ax1, ax2 = axes
    # ===== 左图 =====
    sc = ax1.scatter(
        coords[:, 0],
        coords[:, 1],
        c=pseudotime,
        cmap="viridis",
        s=8,
        alpha=0.9
    )

    ax1.set_title("Pseudotime", fontsize=14)
    ax1.set_xlabel("UMAP1")
    ax1.set_ylabel("UMAP2")

    # 保持等比例
    ax1.set_aspect("equal", adjustable="box")
    ax2.set_aspect("equal", adjustable="box")

    # ----------------------------
    # 3️⃣ 优化 colorbar（不压缩主图）
    # ----------------------------
    divider = make_axes_locatable(ax1)
    cax = divider.append_axes("right", size="3%", pad=0.05)
    cbar = plt.colorbar(sc, cax=cax)
    cbar.set_label("Pseudotime", rotation=270, labelpad=15)
    cbar.ax.tick_params(labelsize=9)

    # ===== 右图 =====
    ax2.scatter(
        coords[:, 0],
        coords[:, 1],
        c="lightgrey",
        s=5,
        alpha=0.3
    )

    ax2.set_title("Cluster-level Directional Flow", fontsize=14)
    ax2.set_xlabel("UMAP1")

    # 画 centroid
    ax2.scatter(
        cluster_stats["x_mean"],
        cluster_stats["y_mean"],
        s=300,
        c="white",
        edgecolor="black",
        zorder=3
    )

    for _, row in cluster_stats.iterrows():
        ax2.text(
            row["x_mean"],
            row["y_mean"],
            row["cluster"],
            ha="center",
            va="center",
            fontsize=9,
            fontweight="bold",
            zorder=4
        )
    # ----------------------------
    # 4️⃣ 画方向箭头
    # ----------------------------
    for i in range(len(cluster_stats)):
        for j in range(len(cluster_stats)):
            if i == j:
                continue
            pt_i = cluster_stats.iloc[i]["pt_mean"]
            pt_j = cluster_stats.iloc[j]["pt_mean"]
            if pt_i >= pt_j:
                continue
            xi, yi = cluster_stats.iloc[i][["x_mean", "y_mean"]]
            xj, yj = cluster_stats.iloc[j][["x_mean", "y_mean"]]
            dist = np.sqrt((xi - xj) ** 2 + (yi - yj) ** 2)
            if dist > distance_threshold:
                continue
            arrow = FancyArrowPatch(
                (xi, yi),
                (xj, yj),
                connectionstyle="arc3,rad=0.2",
                arrowstyle="-|>",
                mutation_scale=arrow_scale,
                linewidth=2,
                color="black",
                alpha=0.8
            )
            ax2.add_patch(arrow)
    # ----------------------------
    # 5️⃣ 统一坐标范围
    # ----------------------------
    xmin, xmax = coords[:, 0].min(), coords[:, 0].max()
    ymin, ymax = coords[:, 1].min(), coords[:, 1].max()
    ax1.set_xlim(xmin, xmax)
    ax1.set_ylim(ymin, ymax)
    # ----------------------------
    # 6️⃣ 让 UMAP 占满画布
    # ----------------------------
    plt.subplots_adjust(
        left=0.06,
        right=0.97,
        bottom=0.08,
        top=0.92,
        wspace=0.25
    )
    fig_dir ='./TabulaStromal_figures/Fibroblasts/tragectory_fig/'
    # plt.savefig(
    # fig_dir+"Fibro_CellRank_DirectionFlow.pdf",
    # dpi=300, bbox_inches='tight',format='pdf')
    # plt.savefig(fig_dir+"Fibro_CellRank_DirectionFlow.png", dpi=300, bbox_inches="tight")
    plt.show()
    

In [ ]:
# dir()


In [ ]:
cluster_directional_flow_optimized(
    adata,
    cluster_key="Celltype_general",
    pseudotime_key="dpt_pseudotime",
    min_cells=50,
    distance_threshold=10,
    arrow_scale=18,
    figsize=(12, 5)
)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from matplotlib.gridspec import GridSpec
from scipy.spatial.distance import cdist
from mpl_toolkits.axes_grid1 import make_axes_locatable

def cluster_level_directional_flow_celltype(adata):
    
    if 'dpt_pseudotime' not in adata.obs.columns:
        print("没有伪时间数据")
        return
    
    coords = adata.obsm['X_umap']
    pseudotime = adata.obs['dpt_pseudotime'].values
    clusters = adata.obs['leiden'].astype(str).values
    celltypes = adata.obs['Celltype_general'].astype(str).values
    # ===============================
    # 1️⃣ 计算 cluster-level 中心 & 平均伪时间
    # ===============================
    cluster_df = pd.DataFrame({
        "cluster": clusters,
        "pt": pseudotime,
        "x": coords[:, 0],
        "y": coords[:, 1]
    })
    cluster_stats = (
        cluster_df
        .groupby("cluster")
        .agg({
            "pt": "mean",
            "x": "mean",
            "y": "mean"
        })
        .reset_index()
    )

    centers = cluster_stats[["x", "y"]].values
    pt_means = cluster_stats["pt"].values
    cluster_labels = cluster_stats["cluster"].values

    # ===============================
    # 2️⃣ 计算方向箭头（低pt → 高pt）
    # ===============================
    dist_matrix = cdist(centers, centers)
    np.fill_diagonal(dist_matrix, np.inf)

    arrows = []
    for i in range(len(cluster_labels)):
        # 找到最近且伪时间更高的cluster
        candidates = np.where(pt_means > pt_means[i])[0]
        if len(candidates) == 0:
            continue
        nearest = candidates[np.argmin(dist_matrix[i, candidates])]
        arrows.append((i, nearest))

    # ===============================
    # 3️⃣ Celltype 颜色
    # ===============================
    unique_celltypes = sorted(np.unique(celltypes))
    n_types = len(unique_celltypes)

    cmap = plt.cm.get_cmap("tab20", n_types)
    celltype_color_map = {
        ct: cmap(i) for i, ct in enumerate(unique_celltypes)
    }

    cell_colors = [celltype_color_map[ct] for ct in celltypes]

    # ===============================
    # 4️⃣ 画图（等比例）
    # ===============================
    fig = plt.figure(figsize=(14, 6))
    gs = GridSpec(1, 2, width_ratios=[1, 1])

    ax1 = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1], sharex=ax1, sharey=ax1)

    # ===============================
    # 左图：Pseudotime
    # ===============================
    sc1 = ax1.scatter(
        coords[:, 0],
        coords[:, 1],
        c=pseudotime,
        cmap="viridis",
        s=8,
        alpha=0.85,
        linewidth=0
    )

    ax1.set_title("Pseudotime", fontsize=14, fontweight="bold")
    # cbar = plt.colorbar(sc1, ax=ax1, fraction=0.046, pad=0.04)
    # cbar.set_label("Pseudotime")

    # 3️⃣ 优化 colorbar（不压缩主图）
    divider = make_axes_locatable(ax1)
    cax = divider.append_axes("right", size="3%", pad=0.05) 
    cbar = plt.colorbar(sc1, cax=cax)
    cbar.set_label("Pseudotime", rotation=270, labelpad=15)
    cbar.ax.tick_params(labelsize=9)
    
    # ===============================
    # 右图：Celltype + Direction
    # ===============================
    ax2.scatter(
        coords[:, 0],
        coords[:, 1],
        c=cell_colors,
        s=8,
        alpha=0.9,
        linewidth=0
    )

    coords = adata.obsm["X_umap"]
    cluster_key="Celltype_general"
    pseudotime_key="dpt_pseudotime"
    min_cells=30
    distance_threshold=8
    arrow_scale=15
    pseudotime = adata.obs[pseudotime_key].values
    clusters = adata.obs[cluster_key]
    df = pd.DataFrame({
        "x": coords[:, 0],
        "y": coords[:, 1],
        "pseudotime": pseudotime,
        "cluster": clusters.values
    })
    # ----------------------------
    # 1️⃣ 计算 cluster stats
    # ----------------------------
    cluster_stats_1 = (
        df.groupby("cluster")
        .agg(
            x_mean=("x", "mean"),
            y_mean=("y", "mean"),
            pt_mean=("pseudotime", "mean"),
            size=("cluster", "size")
        )
        .reset_index()
    )
    cluster_stats_1 = cluster_stats_1[cluster_stats_1["size"] > min_cells]
    for _, row in cluster_stats_1.iterrows():
        ax2.text(
            row["x_mean"],
            row["y_mean"],
            row["cluster"],
            ha="center",
            va="center",
            fontsize=9,
            fontweight="bold",
            zorder=4
        )
    # ----------------------------
    # 4️⃣ 画方向箭头
    # ----------------------------
    for i in range(len(cluster_stats_1)):
        for j in range(len(cluster_stats_1)):
            if i == j:
                continue
            pt_i = cluster_stats_1.iloc[i]["pt_mean"]
            pt_j = cluster_stats_1.iloc[j]["pt_mean"]
            if pt_i >= pt_j:
                continue
            xi, yi = cluster_stats_1.iloc[i][["x_mean", "y_mean"]]
            xj, yj = cluster_stats_1.iloc[j][["x_mean", "y_mean"]]
            dist = np.sqrt((xi - xj) ** 2 + (yi - yj) ** 2)
            if dist > distance_threshold:
                continue
            arrow = FancyArrowPatch(
                (xi, yi),
                (xj, yj),
                connectionstyle="arc3,rad=0.2",
                arrowstyle="-|>",
                mutation_scale=arrow_scale,
                linewidth=2,
                color="black",
                alpha=0.8
            )
            ax2.add_patch(arrow)
            
    ax2.set_title("Celltype + Directional Flow", fontsize=14, fontweight="bold")

    # ===============================
    # 5️⃣ 美化（等比例）
    # ===============================
    for ax in [ax1, ax2]:
        ax.set_aspect("equal", adjustable="box")
        ax.set_xlabel("UMAP1")
        ax.set_ylabel("UMAP2")
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    # ===============================
    # 6️⃣ 精简 legend（避免压缩 UMAP）
    # ===============================
    legend_elements = [
        Line2D(
            [0], [0],
            marker='o',
            color='w',
            label=ct,
            markerfacecolor=celltype_color_map[ct],
            markersize=6
        )
        for ct in unique_celltypes
    ]

    ax2.legend(
        handles=legend_elements,
        title="Celltype_general",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        frameon=False,
        fontsize=8,
        title_fontsize=9
    )
    plt.subplots_adjust(
        left=0.06,
        right=0.97,
        bottom=0.08,
        top=0.92,
        wspace=0.25
    )
    
    plt.tight_layout()
    fig_dir ='./TabulaStromal_figures/Fibroblasts/tragectory_fig/'
    # plt.savefig(
    # fig_dir+"Fibro_CellRank_Celltype_DirectionFlow.pdf",
    # dpi=300, bbox_inches='tight',format='pdf')
    # plt.savefig(fig_dir+"Fibro_CellRank_Celltype_DirectionFlow.png", dpi=300, bbox_inches="tight")
    plt.show()
    

In [ ]:
cluster_level_directional_flow_celltype(adata)


In [ ]:
# cluster_stats

In [ ]:
sc.settings.set_figure_params(dpi=80, frameon=False, figsize=(3, 3), facecolor="white")
sc.pl.umap(adata, color=['Celltype'] ) 


In [ ]:
ck = cr.kernels.ConnectivityKernel(adata)
ck.compute_transition_matrix()


In [ ]:
# help(kernel.plot_projection)


In [ ]:
################ 用Cellrank 的 GPCCA 来构建connectivity Kernel
g = cr.estimators.GPCCA(ck)
g.compute_schur(n_components=20)


In [ ]:
g.compute_macrostates(n_states=4)
g.plot_macrostates(which='all',basis="umap")


In [ ]:
# help(g.plot_macrostates)


In [ ]:
# initial_mask = adata.obs["Celltype_general"] == "qFibro"
# g.set_initial_states(initial_mask)


In [ ]:
g.predict_terminal_states(method='top_n',n_states=3,allow_overlap=True)
# g.plot_terminal_states(basis="umap")


In [ ]:
#################### 基于Cellrank 用另一种策略做轨迹分析 
cr.__version__
# 1️⃣ 构建 kernel
ck = cr.kernels.ConnectivityKernel(adata)
# 2️⃣ 计算 transition matrix
ck.compute_transition_matrix()


In [ ]:
ck.plot_projection(
    basis="umap",
    color="Celltype",   # 或者 age_group
    legend_loc="right margin"
)


In [ ]:
#### 可视化方向( Kernel ) 投影 —— 方向场
ck.plot_projection(
    basis="umap",
    color="Celltype_general",   # 或者 age_group
    legend_loc="right margin"
)


In [ ]:
g = cr.estimators.GPCCA(ck)    # 构建GPCCA Estimator


In [ ]:
## Schur分解(降维)
g.compute_schur(n_components=20)


In [ ]:
## 识别 Macrostates
g.compute_macrostates(n_states=4)


In [ ]:
g.plot_macrostates(
    basis="umap",
    which='all',
    legend_loc="right margin"
) 


In [ ]:
# initial_mask = adata.obs["Celltype_general"] == "qFibro"
# g.set_initial_states(initial_mask)


In [ ]:
###### 这个方法似乎可以手动设置 initial_states & terminal_states , 但 g 没有compute_terminal_states() 函数，也没有 plot_terminal_states() 函数
g.predict_initial_states(allow_overlap=True)
g.predict_terminal_states(allow_overlap=True)
# g.plot_terminal_states(basis="umap")


In [ ]:


g.compute_fate_probabilities()


In [ ]:
g.plot_fate_probabilities(
    basis="umap",
    same_plot=True,   # 每个 terminal 单独一张
    legend_loc="right margin"
)


In [ ]:
# help(g.plot_fate_probabilities)
##### 计算 Absorption time( 真正的 Pseudotime)
g.compute_absorption_times()


In [ ]:
sc.pl.scatter(
    adata,
    x='0',
     y='Celltype_general'
   #  color="Age_group"
)


In [ ]:
adata.obs.columns

In [ ]:
sc.pl.umap(
    adata,
    color=g.fate_probabilities.names,
    cmap="viridis"
)


In [ ]:
sc.pl.umap(
    adata,
    color=[0,1,2,3],
    cmap="viridis"
)


In [ ]:
cr.pl.streamline(
    ck,
    basis="umap",
    color="Celltype"
)


In [ ]:
adata.obsm["lineages_fwd"]


In [ ]:
drivers = g.compute_lineage_drivers()

drivers.head()


In [ ]:
for key in ['0','1','2','3','qFibro','iFibro','eFibro','Myfibro']:
    print(key)
    g.plot_lineage_drivers(lineage=key ,ncols=2, n_genes=10)    # ["qFibro",'eFibro','iFibro','Myfibro']
    fig_dir ='./TabulaStromal_figures/Fibroblasts/tragectory_fig/'
    plt.savefig(
    fig_dir+"Fibro_CellRank_Celltype_DriverGene_state_"+key+".pdf",
    dpi=300, bbox_inches='tight',format='pdf')
    plt.savefig(fig_dir+"Fibro_CellRank_Celltype_DriverGene_state_"+key+".png", dpi=300, bbox_inches="tight")
    plt.show()



In [ ]:
g.plot_lineage_drivers(lineage="qFibro", n_genes=10)


In [ ]:
help(g.plot_lineage_drivers)


In [ ]:
# sorted(dir())


In [ ]:
# cr.pl.circular_projection(adata, keys=[0,1,2,3])


In [ ]:
adata.shape

In [ ]:
# sorted(dir())


In [ ]:
g._absorption_times


In [ ]:
################# 没有 g.plot_absorption_times 函数
# g.plot_absorption_times(
#     basis="umap",
#     cmap="viridis"
# )


In [ ]:
adata.obs[['0', '1', '2', '3']] = g.absorption_times   # 将结果保存


In [ ]:
sc.pl.violin(
    adata,
    keys=g.fate_probabilities.names,
    groupby="Celltype_general",
    use_raw=False
)


In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import NullFormatter
# 获取要展示的基因/特征
keys = g.fate_probabilities.names
n_keys = len(keys)
# 计算最佳布局
n_cols = min(4, n_keys)  # 最多4列
n_rows = int(np.ceil(n_keys / n_cols))
# 创建大图
fig = plt.figure(figsize=(5*n_cols, 4*n_rows))
# 为每个keys创建子图
for idx, key in enumerate(keys):
    # 计算子图位置
    row = idx // n_cols
    col = idx % n_cols
    position = row * n_cols + col + 1
    ax = plt.subplot(n_rows, n_cols, position)
    # 绘制小提琴图
    sc.pl.violin(adata, 
                keys=key,
                groupby="Celltype_general",
                use_raw=False,
                ax=ax,
                show=False,
                stripplot=False)  # 不显示散点
    # 设置标题
    ax.set_title(key, fontsize=12, fontweight='bold')
    # 处理x轴标签
    if row == n_rows - 1:  # 最后一行
        # 旋转x轴标签
        ax.tick_params(axis='x', rotation=45, labelsize=9)
        ax.set_xlabel("Celltype_general", fontsize=10)
    else:
        # 隐藏x轴标签
        ax.set_xlabel("")
        ax.tick_params(axis='x', labelbottom=False)
    # 美化y轴标签
    ax.set_ylabel("Expression", fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
adata.obs.columns
